In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install fastdtw karateclub holidays

In [3]:
import numpy as np
import pandas as pd
from holidays import country_holidays

In [ ]:
CSV_PATH = "tempCleanedDataForHieu.csv"

df = pd.read_csv(CSV_PATH)

df["pickup_ts"]  = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["pu_lon"]  = pd.to_numeric(df["pickup_longitude"], errors="coerce")
df["pu_lat"] = pd.to_numeric(df["pickup_latitude"], errors="coerce")

df = df[["pickup_ts","pu_lon","pu_lat"]].dropna()
df.head()

## TimeStep 30mins + grid 20x20

In [ ]:
INTERVAL_MIN = 30
H, W = 20, 20
S = 7

In [ ]:
df["bucket"] = df["pickup_ts"].dt.floor(f"{INTERVAL_MIN}min")

lon_min, lon_max = df["pu_lon"].min(), df["pu_lon"].max()
lat_min, lat_max = df["pu_lat"].min(), df["pu_lat"].max()

dlon = (lon_max - lon_min) / W
dlat = (lat_max - lat_min) / H

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def lonlat_to_cell(lon, lat):
    x = int(np.floor((lon - lon_min) / dlon))
    y = int(np.floor((lat - lat_min) / dlat))
    x = clamp(x, 0, W-1)
    y = clamp(y, 0, H-1)
    return x, y

xy = df.apply(lambda r: lonlat_to_cell(r["pu_lon"], r["pu_lat"]), axis=1)
df["x"] = [p[0] for p in xy]
df["y"] = [p[1] for p in xy]

df.head()

In [ ]:
demand_tbl = (df.groupby(["bucket","y","x"])
                .size()
                .rename("demand")
                .reset_index())

all_buckets = pd.date_range(demand_tbl["bucket"].min(),
                            demand_tbl["bucket"].max(),
                            freq=f"{INTERVAL_MIN}min")
t2i = {b: i for i, b in enumerate(all_buckets)}

X = np.zeros((len(all_buckets), H, W), dtype=np.float32)
for r in demand_tbl.itertuples(index=False):
    t = t2i[pd.to_datetime(r.bucket)]
    X[t, int(r.y), int(r.x)] = float(r.demand)

## Train/Test split + Min-max norm

In [ ]:
train_ratio = 0.8
T = X.shape[0]
t_split = int(T * train_ratio)

X_train = X[:t_split]
X_test = X[t_split:]

In [ ]:
mn, mx = X_train.min(), X_train.max()
Xn = (X - mn) / (mx - mn + 1e-8)

In [ ]:
us_holidays = country_holidays("US")

def make_context_features(buckets: pd.DatetimeIndex):
    minutes = buckets.hour * 60 + buckets.minute
    tod = minutes / (24*60)

    sin_t = np.sin(2*np.pi*tod)
    cos_t = np.cos(2*np.pi*tod)

    dow = buckets.dayofweek.values
    dow_oh = np.eye(7)[dow].astype(np.float32)

    hol = np.array([1 if b.date() in us_holidays else 0 for b in buckets], dtype=np.int32)
    hol_oh = np.eye(2)[hol].astype(np.float32)

    ctx = np.concatenate([sin_t[:,None], cos_t[:,None], dow_oh, hol_oh], axis=1).astype(np.float32)
    return ctx

In [ ]:
global_mean = X.mean(axis=(1,2))
mean4 = np.convolve(global_mean, np.ones(4)/4, mode="same")

mean4_train = mean4[:t_split]
m_mn, m_mx = mean4_train.min(), mean4_train.max()
mean4_n = (mean4 - m_mn) / (m_mx - m_mn + 1e-8)
mean4_n = mean4_n.astype(np.float32)

## Spatial view - zero padding

In [ ]:
PAD = S // 2

def get_patch(grid_2d, y, x, S=7):
    H, W = grid_2d.shape
    pad = S//2
    padded = np.pad(grid_2d, ((pad,pad),(pad,pad)), mode="constant", constant_values=0)
    y2, x2 = y + pad, x + pad
    patch = padded[y2-pad:y2+pad+1, x2-pad:x2+pad+1]
    return patch.astype(np.float32)

## Semantic View

In [ ]:
def build_weekly_profile(X, all_buckets, interval_min=30):

    T, H, W = X.shape
    b = pd.DatetimeIndex(all_buckets)

    steps_per_day = int(24 * 60 / interval_min)
    P = 7 * steps_per_day
    R = H * W

    idx = (b.dayofweek.values * steps_per_day + (b.hour.values * (60//interval_min) + (b.minute.values//interval_min))).astype(int)

    profiles = np.zeros((R, P), dtype=np.float32)
    counts   = np.zeros((R, P), dtype=np.int32)

    flat_X = X.reshape(T, R)
    for t in range(T):
        k = idx[t]
        profiles[:, k] += flat_X[t]
        counts[:, k] += 1

    profiles = profiles / np.maximum(counts, 1)
    return profiles

In [ ]:
from fastdtw import fastdtw

def dtw_distance(a, b):
    dist, _ = fastdtw(a, b)
    return dist

##### paper dùng 400x400 -> demo 100

In [ ]:
import networkx as nx

def build_semantic_graph(profiles, topk=5, subset=None):

    R = profiles.shape[0]
    nodes = np.arange(R) if subset is None else np.array(subset, dtype=int)

    G = nx.Graph()
    G.add_nodes_from(nodes.tolist())

    for i in nodes:
        dists = []
        for j in nodes:
            if i == j:
                continue
            dist = dtw_distance(profiles[i], profiles[j])
            dists.append((j, dist))

        dists.sort(key=lambda x: x[1])
        for (j, dist) in dists[:topk]:
            w = 1.0 / (dist + 1e-6)
            G.add_edge(int(i), int(j), weight=float(w))

    return G

In [ ]:
subset = np.arange(100)
G = build_semantic_graph(profiles, topk=5, subset=subset)